In [52]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report


In [53]:

# ============================================================
# 1. Load CSV (unchanged)
# ============================================================

SEED = 42
EPOCHS = 200

np.random.seed(SEED)
torch.manual_seed(SEED)


In [54]:

df = pd.read_csv("cyber_dataset_C.csv")


In [55]:

df


,prompt,tool_ground_truth,tool_predicted,pred_svm,source,row_id
0,The vendor gave us a jump-box address and temp...,SSHConnect,SSHConnect,SSHConnect,pilot,C0000
1,Can you establish a terminal session to the Ub...,SSHConnect,SSHConnect,SSHConnect,pilot,C0001
2,We moved the inventory appliance to a new subn...,NmapScan,NmapScan,NmapScan,pilot,C0002
3,Does 172.16.9.25 still advertise that old SSH ...,NmapScan,NmapScan,NmapScan,pilot,C0003
4,The change request says the database node shou...,PortScan,PortScan,PortScan,pilot,C0004
...,...,...,...,...,...,...
1495,Read the local journal around the fan-control ...,ReadSyslog,ReadAuthLog,ReadSyslog,batch495,C1510
1496,The VPN interface renamed itself on this lapto...,ReadSyslog,GetSystemInfo,ReadSyslog,batch495,C1511
1497,After we rolled out a new eBPF-based observabi...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1512
1498,Get the operating-system journal entries from ...,ReadSyslog,ReadSyslog,ReadSyslog,batch495,C1513


In [56]:

encoder = LabelEncoder()

y = encoder.fit_transform(df["tool_ground_truth"])


In [57]:

num_classes = len(encoder.classes_)

num_classes 


11

In [58]:

class_to_id = {
    name: i for i, name in enumerate(encoder.classes_)
}

class_to_id


{'BlockIP': 0,
 'CheckFailedLogins': 1,
 'CheckVulnerability': 2,
 'GetSystemInfo': 3,
 'ListProcesses': 4,
 'ListeningPorts': 5,
 'NmapScan': 6,
 'PortScan': 7,
 'ReadAuthLog': 8,
 'ReadSyslog': 9,
 'SSHConnect': 10}

In [59]:

y_svm = np.array([
    class_to_id[name]
    for name in df["pred_svm"]
])



y_svm.shape





(1500,)

In [60]:

y_svm


array([10, 10,  6, ...,  9,  9,  9])

In [61]:

y_llama = np.array([
    class_to_id.get(name, -1)
    for name in df["tool_predicted"]
])


In [62]:

# Unknown is encoded as -1.

set ( [  name for name in df["tool_predicted"]   ]  )


{'BlockIP',
 'CheckFailedLogins',
 'CheckVulnerability',
 'GetSystemInfo',
 'ListProcesses',
 'ListeningPorts',
 'NmapScan',
 'PortScan',
 'ReadAuthLog',
 'ReadSyslog',
 'SSHConnect',
 'Unknown'}

In [63]:

print(np.sum(y_llama == -1))


14


In [64]:

# ============================================================
# 2. Train / validation / test split
# ============================================================

indices = np.arange(len(df))

indices


array([   0,    1,    2, ..., 1497, 1498, 1499])

In [65]:

random_seed = int( random.random() * 100 )     ## 42

## X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)

trainval_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=random_seed,             ## SEED,
    stratify=y
)


In [66]:

print(  len(trainval_idx)  )
print(   len(test_idx   )  ) 


1200
300


In [67]:

test_idx


array([1348, 1221,  607,  720,  900,  525,  509, 1180, 1430, 1274,  650,
        919, 1099,  282, 1412, 1234,  475, 1267,  469, 1035,  958, 1410,
       1453,  319,  818,  538, 1296,  187,  191,  329, 1385, 1393, 1284,
        307,  363,  164,  292, 1358,  881, 1044,    7,  354,   84,  395,
       1028,  180, 1161,  270,  542,  629,  298,  491, 1089, 1422,    4,
        381,  481, 1473,  938,  142,   71,  117, 1112,  585,  954,  281,
         54,  809,  964,  876, 1115,  987,  169,  899,  217, 1032,  844,
       1066,  547, 1440,  385,  355,  760,  784, 1310, 1158,  597, 1056,
        729,  568,  989,  753,  457,  130,  336, 1049,  769,  337,  495,
       1191,    3, 1490, 1189,  424, 1425,  648,  280,  569,  489, 1303,
       1012,  800,  514,  340,  446,  880,  649, 1432,  248, 1093,  672,
       1382, 1265,  680, 1096,  128, 1165, 1318,  299,  659,   53, 1227,
        757,  950,  731, 1230, 1389,  593,  156, 1181,  922, 1206,  651,
        500,  546,  583,  396,  756,  824,  762,  3

In [68]:

train_idx = trainval_idx


In [69]:

print("Training:", len(train_idx))
print("Testing:", len(test_idx))



Training: 1200
Testing: 300


In [70]:

# ============================================================
# 3. Features: SVM + Llama predictions (23 features)
# ============================================================

# SVM has 11 classes
svm_onehot = np.zeros((len(df), num_classes))

svm_onehot.shape


(1500, 11)

In [71]:

y_svm


array([10, 10,  6, ...,  9,  9,  9])

In [72]:

valid_svm = y_svm >= 0

print(  np.sum( valid_svm )  )
print(     len( valid_svm )  )


1500
1500


In [73]:


svm_onehot[ 
    np.where(valid_svm)[0], y_svm[valid_svm] 
] = 1


In [74]:

for thing in svm_onehot:
    print(  thing  )


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 0

In [75]:

svm_onehot.shape


(1500, 11)

In [76]:

# Llama has 11 classes + Unknown
llama_onehot = np.zeros((len(df), num_classes + 1))

llama_onehot.shape


(1500, 12)

In [77]:

llama_idx = np.where(y_llama >= 0, y_llama, num_classes)


In [78]:


print(     len(  llama_idx )  )


1500


In [79]:

'''
import numpy as np

y_llama_rc = np.array([2, -1, 4, 0, -1])
num_classes_rc = 11

llama_idx_rc = np.where(y_llama_rc >= 0, y_llama_rc, num_classes_rc)

print("Original:", y_llama_rc)
print("Modified:", llama_idx_rc)

'''


'\nimport numpy as np\n\ny_llama_rc = np.array([2, -1, 4, 0, -1])\nnum_classes_rc = 11\n\nllama_idx_rc = np.where(y_llama_rc >= 0, y_llama_rc, num_classes_rc)\n\nprint("Original:", y_llama_rc)\nprint("Modified:", llama_idx_rc)\n\n'

In [80]:


llama_onehot[ 
      np.arange(len(df)), llama_idx
] = 1


In [81]:

for thing in llama_onehot:
    print( thing )


[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 1.

In [82]:


X = np.hstack((svm_onehot, llama_onehot)).astype(np.float32)

X.shape


(1500, 23)

In [83]:

X_train = torch.tensor( X[train_idx] )
X_test  = torch.tensor( X[test_idx]  )

print("Input features:", X.shape[1])


Input features: 23


In [84]:

print(  X_train.shape  )
print(  X_test.shape   )


torch.Size([1200, 23])
torch.Size([300, 23])


In [85]:

# ============================================================
# 4. Neural network targets (11 cybersecurity tools)
# ============================================================

target = y

target.shape


(1500,)

In [86]:


target_train = torch.tensor(
    target[train_idx],
    dtype=torch.long
)



In [87]:

target_train.shape


torch.Size([1200])

In [88]:

target_train


tensor([2, 6, 9,  ..., 0, 0, 4])

In [89]:

# ============================================================
# 5. Neural network (11 cybersecurity tools)
# ============================================================

class SelectorNet(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 11)
        )

    def forward(self, x):
        return self.net(x)


In [90]:



model = SelectorNet(  X.shape[1]  )




In [91]:

X.shape[1]


23

In [92]:

criterion = nn.CrossEntropyLoss()


In [93]:

optimizer = optim.Adam(model.parameters(), lr=0.001)


In [94]:



# ============================================================
# Train
# ============================================================

for epoch in range(EPOCHS):

    model.train()
    optimizer.zero_grad()

    logits = model(  X_train  )

    loss = criterion(logits, target_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Loss = {loss.item():.4f}"
        )





Epoch  20 | Loss = 2.3678
Epoch  40 | Loss = 2.3098
Epoch  60 | Loss = 2.2137
Epoch  80 | Loss = 2.0579
Epoch 100 | Loss = 1.8286
Epoch 120 | Loss = 1.5317
Epoch 140 | Loss = 1.2134
Epoch 160 | Loss = 0.9109
Epoch 180 | Loss = 0.6585
Epoch 200 | Loss = 0.4731


In [95]:

model.eval()

with torch.no_grad():
    ensemble_pred_res = torch.argmax(model(X_test), dim=1).numpy()


In [96]:

ensemble_pred_res.shape


(300,)

In [97]:

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_true = y[test_idx]



In [103]:


print("Accuracy:", accuracy_score(y_true, ensemble_pred_res))

print("Macro F1:", f1_score(      y_true, ensemble_pred_res, average="macro" ))

print("Confusion Matrix:")
print(           confusion_matrix(y_true, ensemble_pred_res))

print("Classification Report:")
print(classification_report(      y_true, ensemble_pred_res,  zero_division=0   ))


Accuracy: 0.9533333333333334
Macro F1: 0.9528507498923147
Confusion Matrix:
[[28  0  0  0  0  0  0  0  0  0  0]
 [ 0 27  0  0  0  0  0  0  0  0  0]
 [ 0  0 26  0  0  0  0  1  0  0  0]
 [ 0  0  1 27  0  0  0  0  0  0  0]
 [ 1  0  0  0 23  2  0  1  0  0  0]
 [ 0  0  1  0  0 23  1  1  1  0  0]
 [ 0  0  2  0  0  0 25  0  0  0  0]
 [ 0  0  0  0  0  1  0 26  0  0  0]
 [ 0  0  0  0  0  0  0  0 27  0  0]
 [ 0  0  0  0  1  0  0  0  0 26  0]
 [ 0  0  0  0  0  0  0  0  0  0 28]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        28
           1       1.00      1.00      1.00        27
           2       0.87      0.96      0.91        27
           3       1.00      0.96      0.98        28
           4       0.96      0.85      0.90        27
           5       0.88      0.85      0.87        27
           6       0.96      0.93      0.94        27
           7       0.90      0.96      0.93        27
           8     

In [104]:

print("Accuracy:", accuracy_score(y_true, y_svm[test_idx]))

print("Macro F1:", f1_score(      y_true, y_svm[test_idx], average="macro" ))

print("Confusion Matrix:")
print(           confusion_matrix(y_true, y_svm[test_idx]))

print("Classification Report:")
print(classification_report(      y_true, y_svm[test_idx],  zero_division=0   ))


Accuracy: 0.9433333333333334
Macro F1: 0.9427740676208117
Confusion Matrix:
[[27  0  0  0  0  0  0  0  1  0  0]
 [ 0 25  0  1  0  0  0  0  1  0  0]
 [ 0  0 25  0  0  0  1  0  0  0  1]
 [ 0  0  1 27  0  0  0  0  0  0  0]
 [ 0  0  0  0 24  1  2  0  0  0  0]
 [ 0  0  0  1  1 22  1  1  1  0  0]
 [ 0  0  0  0  0  0 27  0  0  0  0]
 [ 0  0  0  0  0  1  0 26  0  0  0]
 [ 0  0  0  0  0  0  0  0 26  1  0]
 [ 0  0  0  0  1  0  0  0  0 26  0]
 [ 0  0  0  0  0  0  0  0  0  0 28]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98        28
           1       1.00      0.93      0.96        27
           2       0.96      0.93      0.94        27
           3       0.93      0.96      0.95        28
           4       0.92      0.89      0.91        27
           5       0.92      0.81      0.86        27
           6       0.87      1.00      0.93        27
           7       0.96      0.96      0.96        27
           8     

In [105]:

print("Accuracy:", accuracy_score(y_true, y_llama[test_idx]  ))

print("Macro F1:", f1_score(      y_true, y_llama[test_idx], average="macro" ))

print("Confusion Matrix:")
print(           confusion_matrix(y_true, y_llama[test_idx]   ))

print("Classification Report:")
print(classification_report(      y_true, y_llama[test_idx]  ,  zero_division=0   ))


Accuracy: 0.8733333333333333
Macro F1: 0.7971537161543131
Confusion Matrix:
[[ 0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 28  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 26  0  0  0  0  1  0  0  0  0]
 [ 0  0  0 26  0  0  0  0  1  0  0  0]
 [ 0  0  0  0 28  0  0  0  0  0  0  0]
 [ 1  0  1  0  3 19  2  0  1  0  0  0]
 [ 0  0  0  1  0  0 11  3 11  1  0  0]
 [ 0  0  0  2  0  0  0 25  0  0  0  0]
 [ 1  0  0  0  0  0  0  8 18  0  0  0]
 [ 0  0  0  0  0  0  0  0  0 27  0  0]
 [ 0  0  0  0  1  0  0  0  0  0 26  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 28]]
Classification Report:
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         0
           0       1.00      1.00      1.00        28
           1       0.96      0.96      0.96        27
           2       0.90      0.96      0.93        27
           3       0.88      1.00      0.93        28
           4       1.00      0.70      0.83        27
           5       0.85      0.41      0.55        27

In [106]:

print(f1_score(
    y[test_idx],
    y_llama[test_idx],
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
))


0.8696222358047052


In [107]:

print("Macro F1:", f1_score(
    y_true, y_llama[test_idx],
    labels=np.arange(num_classes),
    average="macro",
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_true, y_llama[test_idx],
    labels=np.arange(num_classes)
))

print("Classification Report:")
print(classification_report(
    y_true, y_llama[test_idx],
    labels=np.arange(num_classes),
    zero_division=0
))


Macro F1: 0.8696222358047052
Confusion Matrix:
[[28  0  0  0  0  0  0  0  0  0  0]
 [ 0 26  0  0  0  0  1  0  0  0  0]
 [ 0  0 26  0  0  0  0  1  0  0  0]
 [ 0  0  0 28  0  0  0  0  0  0  0]
 [ 0  1  0  3 19  2  0  1  0  0  0]
 [ 0  0  1  0  0 11  3 11  1  0  0]
 [ 0  0  2  0  0  0 25  0  0  0  0]
 [ 0  0  0  0  0  0  8 18  0  0  0]
 [ 0  0  0  0  0  0  0  0 27  0  0]
 [ 0  0  0  1  0  0  0  0  0 26  0]
 [ 0  0  0  0  0  0  0  0  0  0 28]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        28
           1       0.96      0.96      0.96        27
           2       0.90      0.96      0.93        27
           3       0.88      1.00      0.93        28
           4       1.00      0.70      0.83        27
           5       0.85      0.41      0.55        27
           6       0.68      0.93      0.78        27
           7       0.58      0.67      0.62        27
           8       0.96      1.00      0.98   


## NIO


In [161]:

## !pip install matplotlib



## NIO - Case 1: Unknown


In [274]:

# 14 cases where Llama predicted Unknown
unknown_idx = np.flatnonzero(y_llama == -1)


In [275]:

unknown_idx


array([ 141,  285,  295,  460,  590,  646, 1081, 1103, 1172, 1184, 1194,
       1205, 1290, 1297])

In [276]:

X_real = torch.tensor(  X[unknown_idx], dtype=torch.float32  )


In [277]:

X_real.shape



torch.Size([14, 23])

In [278]:

X_real[:, 12:]



tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

In [279]:

y_true = torch.tensor( y[unknown_idx], dtype=torch.long)


In [280]:

y_true


tensor([8, 4, 4, 2, 7, 6, 6, 6, 2, 2, 2, 8, 5, 5])

In [281]:

y_true.shape


torch.Size([14])

In [282]:

# Freeze neural network
model.eval()
for p in model.parameters():
    p.requires_grad = False


In [283]:

# NIO: 14 NEW random input vectors (23 features each)
torch.manual_seed(42)

X_nio = torch.rand(14, 23, requires_grad=True)


In [284]:

optimizer = torch.optim.Adam([X_nio], lr=0.01)

loss_fn = nn.CrossEntropyLoss()


In [285]:


for epoch in range(2000):
    
    optimizer.zero_grad()

    output = model(  X_nio  )
    
    ## loss   = nn.CrossEntropyLoss()(output, y_true)
    loss   = loss_fn(  output, y_true   )   

    loss.backward()
    optimizer.step()

    # Keep input features between 0 and 1
    with torch.no_grad():
        X_nio.clamp_(0, 1)
    if epoch % 100 == 0:
            print(epoch, "loss=", loss)




0 loss= tensor(3.2897, grad_fn=<NllLossBackward0>)
100 loss= tensor(0.1722, grad_fn=<NllLossBackward0>)
200 loss= tensor(0.1042, grad_fn=<NllLossBackward0>)
300 loss= tensor(0.0842, grad_fn=<NllLossBackward0>)
400 loss= tensor(0.0748, grad_fn=<NllLossBackward0>)
500 loss= tensor(0.0711, grad_fn=<NllLossBackward0>)
600 loss= tensor(0.0692, grad_fn=<NllLossBackward0>)
700 loss= tensor(0.0682, grad_fn=<NllLossBackward0>)
800 loss= tensor(0.0675, grad_fn=<NllLossBackward0>)
900 loss= tensor(0.0669, grad_fn=<NllLossBackward0>)
1000 loss= tensor(0.0663, grad_fn=<NllLossBackward0>)
1100 loss= tensor(0.0658, grad_fn=<NllLossBackward0>)
1200 loss= tensor(0.0653, grad_fn=<NllLossBackward0>)
1300 loss= tensor(0.0650, grad_fn=<NllLossBackward0>)
1400 loss= tensor(0.0648, grad_fn=<NllLossBackward0>)
1500 loss= tensor(0.0647, grad_fn=<NllLossBackward0>)
1600 loss= tensor(0.0646, grad_fn=<NllLossBackward0>)
1700 loss= tensor(0.0646, grad_fn=<NllLossBackward0>)
1800 loss= tensor(0.0646, grad_fn=<NllLo

In [286]:

# Compare real and NIO-derived Unknown feature (feature 23)

real_disp = X_real[:, 12:].numpy()

real_disp


array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]], dtype=float32)

In [287]:


nio_disp = X_nio[:, 12:].detach().numpy().round(2)
nio_disp


array([[0.23, 0.  , 0.  , 1.  , 0.  , 0.  , 0.48, 1.  , 1.  , 0.25, 0.  ],
       [1.  , 0.15, 1.  , 1.  , 0.62, 0.  , 0.9 , 1.  , 0.  , 0.  , 1.  ],
       [1.  , 0.04, 1.  , 1.  , 0.89, 0.  , 0.94, 1.  , 0.  , 0.  , 1.  ],
       [1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.2 , 1.  , 1.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.78, 1.  , 1.  , 1.  , 0.  , 1.  ],
       [0.75, 0.  , 0.  , 0.  , 0.62, 1.  , 1.  , 0.  , 1.  , 0.43, 1.  ],
       [0.69, 0.  , 0.  , 0.  , 0.67, 1.  , 1.  , 0.  , 1.  , 0.41, 1.  ],
       [0.69, 0.  , 0.  , 0.  , 0.66, 1.  , 1.  , 0.  , 1.  , 0.41, 1.  ],
       [1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.95, 1.  ],
       [1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.92, 1.  ],
       [1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.24, 1.  , 1.  ],
       [0.29, 0.  , 0.  , 1.  , 0.  , 0.  , 0.49, 1.  , 1.  , 0.36, 0.  ],
       [0.  , 0.  , 0.56, 0.2 , 1.  , 1.  , 1.  , 0.46, 0.  , 0.94, 1.  ],
       [0.  , 0.  , 0.57,

In [288]:

nio_disp = X_nio[:, :12].detach().numpy().round(2)
nio_disp


array([[1.  , 0.38, 0.  , 0.  , 1.  , 0.  , 0.  , 0.  , 1.  , 1.  , 1.  ,
        1.  ],
       [0.  , 1.  , 1.  , 1.  , 1.  , 0.91, 0.  , 0.91, 1.  , 0.29, 1.  ,
        0.  ],
       [0.  , 1.  , 1.  , 1.  , 1.  , 0.56, 0.  , 1.  , 1.  , 0.02, 1.  ,
        0.  ],
       [0.  , 1.  , 1.  , 1.  , 0.07, 0.  , 0.  , 0.4 , 0.05, 1.  , 1.  ,
        0.  ],
       [1.  , 0.  , 0.  , 1.  , 0.69, 0.  , 1.  , 1.  , 1.  , 0.  , 0.89,
        0.44],
       [1.  , 0.56, 0.67, 1.  , 0.  , 0.  , 1.  , 1.  , 0.  , 0.  , 0.  ,
        0.  ],
       [1.  , 0.65, 0.66, 1.  , 0.  , 0.  , 1.  , 1.  , 0.  , 0.  , 0.  ,
        0.06],
       [1.  , 0.64, 0.66, 1.  , 0.  , 0.  , 1.  , 1.  , 0.  , 0.  , 0.  ,
        0.06],
       [0.  , 1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.29, 0.26, 1.  , 1.  ,
        0.  ],
       [0.  , 1.  , 1.  , 1.  , 0.  , 0.  , 0.  , 0.28, 0.3 , 1.  , 1.  ,
        0.  ],
       [0.  , 1.  , 1.  , 1.  , 0.11, 0.  , 0.  , 0.43, 0.01, 1.  , 1.  ,
        0.  ],
       [1.  , 0.37, 0

In [289]:

X_nio.shape


torch.Size([14, 23])

In [293]:

real = X_real[:, 22].numpy()

real


array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
      dtype=float32)

In [294]:

nio = X_nio[:, 22].detach().numpy()

nio


array([0.       , 1.       , 1.       , 1.       , 0.9977791, 1.       ,
       1.       , 1.       , 1.       , 1.       , 1.       , 0.       ,
       1.       , 1.       ], dtype=float32)

In [295]:

print("Real Unknown:", real)


Real Unknown: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [296]:

print("NIO Unknown: ", nio)


NIO Unknown:  [0.        1.        1.        1.        0.9977791 1.        1.
 1.        1.        1.        1.        0.        1.        1.       ]


In [297]:

print("Mean absolute error:", np.mean(np.abs(real - nio)))


Mean absolute error: 0.14301577


In [298]:


# Compare all 23 features
print("\nMean squared error (all features):",
      torch.mean((X_real - X_nio.detach())**2).item())



Mean squared error (all features): 0.38009878993034363


In [299]:

12/14


0.8571428571428571